# Notebook 06: Isolated Model Training - Hospital B (Diabetology Focus)

## Objective
Train a local model exclusively on **Hospital B** dataset (which has Diabetes bias).
Evaluate how Hospital B's isolated model performs locally vs on external hospital nodes.



In [1]:
import os
import sys
# Ensure project root is in sys.path for backend and scripts imports
root_path = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if root_path not in sys.path:
    sys.path.insert(0, root_path)
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from backend.ml.model import DiabetesRiskModel

ha = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_A.csv"))
hb = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_B.csv"))
hc = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_C.csv"))

feature_cols = ['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi', 'glucose', 'hba1c', 'cholesterol', 'creatinine']
target_col = 'diabetes'

scaler = StandardScaler()
scaler.fit(pd.concat([ha, hb, hc])[feature_cols])

X_b = scaler.transform(hb[feature_cols].values)
y_b = hb[target_col].values

split_idx = int(len(X_b) * 0.8)
X_b_train, X_b_test = X_b[:split_idx], X_b[split_idx:]
y_b_train, y_b_test = y_b[:split_idx], y_b[split_idx:]

train_loader = DataLoader(TensorDataset(torch.tensor(X_b_train, dtype=torch.float32), torch.tensor(y_b_train, dtype=torch.float32)), batch_size=32, shuffle=True)

model_b = DiabetesRiskModel(input_dim=len(feature_cols))
optimizer = optim.Adam(model_b.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

model_b.train()
for epoch in range(25):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_b(bx).squeeze(), by)
        loss.backward()
        optimizer.step()

print("Hospital B Local Training Complete.")


C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Hospital B Local Training Complete.


## 1. Evaluate Hospital B Model Cross-Node Generalization


In [2]:
def eval_model(model, X_val, y_val):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_val, dtype=torch.float32)).squeeze()
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= 0.5).astype(float)
    return {'acc': accuracy_score(y_val, preds), 'f1': f1_score(y_val, preds, zero_division=0)}

X_a = scaler.transform(ha[feature_cols].values)
y_a = ha[target_col].values
X_c = scaler.transform(hc[feature_cols].values)
y_c = hc[target_col].values

res_b_on_b = eval_model(model_b, X_b_test, y_b_test)
res_b_on_a = eval_model(model_b, X_a, y_a)
res_b_on_c = eval_model(model_b, X_c, y_c)

print("Hospital B Model Results:")
print(f"Local Test (Hospital B):   Acc = {res_b_on_b['acc']:.4f}, F1 = {res_b_on_b['f1']:.4f}")
print(f"Remote Test (Hospital A):  Acc = {res_b_on_a['acc']:.4f}, F1 = {res_b_on_a['f1']:.4f}")
print(f"Remote Test (Hospital C):  Acc = {res_b_on_c['acc']:.4f}, F1 = {res_b_on_c['f1']:.4f}")


Hospital B Model Results:
Local Test (Hospital B):   Acc = 0.8333, F1 = 0.6923
Remote Test (Hospital A):  Acc = 0.9047, F1 = 0.9247
Remote Test (Hospital C):  Acc = 0.8459, F1 = 0.7545


C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
